# APPL ML strategy — execution-ready research notebook

This notebook keeps the ML logic from the supplied `appl.ipynb` and delegates exchange execution to `src/trading`. It does not start an infinite live loop.

In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().resolve().parents[2]))

from src.trading.config import SETTINGS
from src.trading.data import BinanceMarketData
from src.trading.model import causal_predict
from src.trading.strategy import make_signal

SETTINGS

In [ ]:
# Load closed candles
base_url = "https://demo-fapi.binance.com" if SETTINGS.market_type == "FUTURES" else "https://demo-api.binance.com"
market = BinanceMarketData(base_url, SETTINGS.symbol, SETTINGS.bar_size)
df = market.klines(SETTINGS.data_limit)
df.tail()

In [ ]:
# Same causal prediction logic as the original notebook
prediction, timestamp, price = causal_predict(
    df,
    horizon=SETTINGS.horizon,
    lr=0.00325,
    model_l2=0.425,
)
print({"timestamp": timestamp, "price": price, "prediction": prediction})

In [ ]:
# Example signal from a flat state
signal = make_signal(
    pred=prediction,
    timestamp=timestamp,
    price=price,
    previous_position=0,
    previous_notional=0.0,
    equity=SETTINGS.equity,
    risk_per_trade=SETTINGS.risk_per_trade,
    leverage=SETTINGS.leverage,
    threshold_multiplier=SETTINGS.threshold_multiplier,
    taker_fee_rate=SETTINGS.taker_fee_rate,
    slippage_rate=SETTINGS.slippage_rate,
    allow_shorts=SETTINGS.allow_shorts,
)
signal

## Live execution

Use `python run_bot.py` outside Jupyter. The engine reconciles the actual exchange position before/after orders. `ENVIRONMENT=DEMO` is the default and LIVE requires an explicit confirmation flag.